In [ ]:
import pandas as pd
import numpy as np

raw_path = "../data/raw"
processed_path = "../data/processed"

churn = pd.read_csv(f"{raw_path}/ravenstack_churn_events.csv")
accounts = pd.read_csv(f"{raw_path}/ravenstack_accounts.csv")
subscriptions = pd.read_csv(f"{raw_path}/ravenstack_subscriptions.csv")
usage = pd.read_csv(f"{raw_path}/ravenstack_feature_usage.csv")
support = pd.read_csv(f"{raw_path}/ravenstack_support_tickets.csv")

# If feedback_text is not null, it becomes True. Otherwise False.
churn['has_feedback'] = churn['feedback_text'].notnull()

# Aggregate data per account_id

churn_summary = churn.groupby('account_id').agg(
    total_churns=('churn_event_id', 'count'),
    total_refund_usd=('refund_amount_usd', 'sum'),
    last_churn_date=('churn_date', 'max'),
    provided_feedback=('has_feedback', 'max'), # If they ever provided feedback, it stays True
    last_reason=('reason_code', 'last')
).reset_index()

#preview new aggregated table

print("--- AGGREGATED CHURN SUMMARY ---")
print(churn_summary.head())
print(churn_summary.shape)
print(f"\nTotal unique accounts in summary: {len(churn_summary)}")

--- AGGREGATED CHURN SUMMARY ---
  account_id  total_churns  total_refund_usd last_churn_date  \
0   A-00bed1             1             21.17      2024-01-03   
1   A-016043             1             84.75      2024-08-11   
2   A-029f69             3             32.73      2024-12-14   
3   A-02cd81             1              0.00      2024-04-30   
4   A-02fac6             1              0.00      2024-08-21   

   provided_feedback last_reason  
0              False     unknown  
1               True  competitor  
2               True  competitor  
3              False     pricing  
4               True     support  
(352, 6)

Total unique accounts in summary: 352


In [ ]:
import pandas as pd
import numpy as np

# 1. PATHS
raw_path = "../data/raw"
processed_path = "../data/processed"

# 2. LOAD ALL DATA
churn = pd.read_csv(f"{raw_path}/ravenstack_churn_events.csv")
accounts = pd.read_csv(f"{raw_path}/ravenstack_accounts.csv")
subscriptions = pd.read_csv(f"{raw_path}/ravenstack_subscriptions.csv")
usage = pd.read_csv(f"{raw_path}/ravenstack_feature_usage.csv")
support = pd.read_csv(f"{raw_path}/ravenstack_support_tickets.csv")

# ==========================================
# 3. AGGREGATE CHURN EVENTS (The one I missed!)
# ==========================================
churn['has_feedback'] = churn['feedback_text'].notnull()

churn_summary = churn.groupby('account_id').agg(
    total_churns=('churn_event_id', 'count'),
    total_refund_usd=('refund_amount_usd', 'sum'),
    last_churn_date=('churn_date', 'max'),
    provided_feedback=('has_feedback', 'max'),
    last_reason=('reason_code', 'last')
).reset_index()

# ==========================================
# 4. AGGREGATE SUPPORT TICKETS
# ==========================================
support_summary = support.groupby('account_id').agg(
    total_tickets=('ticket_id', 'count'),
    avg_satisfaction=('satisfaction_score', 'mean'), 
    total_escalations=('escalation_flag', 'sum')
).reset_index()

# ==========================================
# 5. AGGREGATE SUBSCRIPTIONS
# ==========================================
subs_summary = subscriptions.groupby('account_id').agg(
    total_mrr=('mrr_amount', 'sum'),
    total_upgrades=('upgrade_flag', 'sum')
).reset_index()

# ==========================================
# 6. AGGREGATE FEATURE USAGE (The Bridge)
# ==========================================
usage_with_account = pd.merge(usage, subscriptions[['subscription_id', 'account_id']], on='subscription_id', how='left')

usage_summary = usage_with_account.groupby('account_id').agg(
    total_usage_count=('usage_count', 'sum'),
    total_usage_duration=('usage_duration_secs', 'sum'),
    total_errors=('error_count', 'sum')
).reset_index()

# ==========================================
# 7. THE MASTER JOIN
# ==========================================
# Start with accounts and attach ALL summaries
master_table = accounts.copy()

master_table = pd.merge(master_table, churn_summary, on='account_id', how='left')
master_table = pd.merge(master_table, support_summary, on='account_id', how='left')
master_table = pd.merge(master_table, subs_summary, on='account_id', how='left')
master_table = pd.merge(master_table, usage_summary, on='account_id', how='left')

# ==========================================
# 8. HANDLE MISSING VALUES (NaNs) FROM LEFT JOINS
# ==========================================

master_table['total_churns'] = master_table['total_churns'].fillna(0)
master_table['total_refund_usd'] = master_table['total_refund_usd'].fillna(0)
master_table['provided_feedback'] = master_table['provided_feedback'].fillna(False)

master_table['total_tickets'] = master_table['total_tickets'].fillna(0)
master_table['total_escalations'] = master_table['total_escalations'].fillna(0)

master_table['total_mrr'] = master_table['total_mrr'].fillna(0)
master_table['total_upgrades'] = master_table['total_upgrades'].fillna(0)

master_table['total_usage_count'] = master_table['total_usage_count'].fillna(0)
master_table['total_usage_duration'] = master_table['total_usage_duration'].fillna(0)
master_table['total_errors'] = master_table['total_errors'].fillna(0)

# ==========================================
# 9. SAVE TO PROCESSED DIRECTORY
# ==========================================
master_table.to_csv(f"{processed_path}/master_features.csv", index=False)

print("--- FINAL MASTER TABLE READY & SAVED ---")
print(f"Shape: {master_table.shape}")
print(master_table.head())

--- FINAL MASTER TABLE READY & SAVED ---
Shape: (500, 23)
  account_id account_name    industry country signup_date referral_source  \
0   A-2e4581    Company_0      EdTech      US  2024-10-16         partner   
1   A-43a9e3    Company_1     FinTech      IN  2023-08-17           other   
2   A-0a282f    Company_2    DevTools      US  2024-08-27         organic   
3   A-1f0ac7    Company_3  HealthTech      UK  2023-08-27           other   
4   A-ce550d    Company_4  HealthTech      US  2024-10-27           event   

    plan_tier  seats  is_trial  churn_flag  ...  provided_feedback  \
0       Basic      9     False       False  ...               True   
1       Basic     18     False        True  ...              False   
2       Basic      1     False       False  ...               True   
3       Basic     24      True       False  ...               True   
4  Enterprise     35     False        True  ...               True   

   last_reason total_tickets  avg_satisfaction total_escal

/var/folders/6z/0zwcwb493_l2wdptb7yp4n2c0000gn/T/ipykernel_4578/1754152999.py:72: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  master_table['provided_feedback'] = master_table['provided_feedback'].fillna(False)


In [8]:
import pandas as pd
import numpy as np

# ==========================================
# 1. PATHS
# ==========================================
raw_path = "../data/raw"
processed_path = "../data/processed"

# ==========================================
# 2. LOAD ALL DATA
# ==========================================
accounts      = pd.read_csv(f"{raw_path}/ravenstack_accounts.csv")
subscriptions = pd.read_csv(f"{raw_path}/ravenstack_subscriptions.csv")
usage         = pd.read_csv(f"{raw_path}/ravenstack_feature_usage.csv")
support       = pd.read_csv(f"{raw_path}/ravenstack_support_tickets.csv")

# ==========================================
# 3. AGGREGATE SUPPORT TICKETS
# ==========================================
support_summary = support.groupby('account_id').agg(
    total_tickets=('ticket_id', 'count'),
    avg_satisfaction=('satisfaction_score', 'mean'),
    total_escalations=('escalation_flag', 'sum')
).reset_index()

# ==========================================
# 4. AGGREGATE SUBSCRIPTIONS
# ==========================================
subs_summary = subscriptions.groupby('account_id').agg(
    total_mrr=('mrr_amount', 'sum'),
    total_upgrades=('upgrade_flag', 'sum')
).reset_index()

# ==========================================
# 5. AGGREGATE FEATURE USAGE (Bridge via subscription_id)
# ==========================================
usage_with_account = pd.merge(
    usage,
    subscriptions[['subscription_id', 'account_id']],
    on='subscription_id',
    how='left'
)

usage_summary = usage_with_account.groupby('account_id').agg(
    total_usage_count=('usage_count', 'sum'),
    total_usage_duration=('usage_duration_secs', 'sum'),
    total_errors=('error_count', 'sum')
).reset_index()

# ==========================================
# 6. MASTER JOIN
# ==========================================
master_table = accounts.copy()
master_table = pd.merge(master_table, support_summary, on='account_id', how='left')
master_table = pd.merge(master_table, subs_summary,   on='account_id', how='left')
master_table = pd.merge(master_table, usage_summary,  on='account_id', how='left')

# ==========================================
# 7. TENURE (Dinamik)
# ==========================================
master_table['signup_date'] = pd.to_datetime(master_table['signup_date'])
reference_date = pd.Timestamp('2025-01-01')  # Dataset snapshot tarihi
master_table['tenure_days'] = (reference_date - master_table['signup_date']).dt.days + 1

# ==========================================
# 8. MISSING VALUE HANDLING
# ==========================================

# Satisfaction: plan_tier'a göre group mean, flag ekle
master_table['provided_satisfaction_score'] = master_table['avg_satisfaction'].notnull()
master_table['avg_satisfaction'] = master_table.groupby('plan_tier')['avg_satisfaction'].transform(
    lambda x: x.fillna(x.mean())
)
# Eğer tüm plan grubunda NaN varsa global mean ile kapat
global_mean_satisfaction = master_table['avg_satisfaction'].mean()
master_table['avg_satisfaction'] = master_table['avg_satisfaction'].fillna(global_mean_satisfaction)

# Geri kalan sayısal sütunları sıfırla doldur
fill_zero_cols = [
    'total_tickets', 'total_escalations',
    'total_mrr', 'total_upgrades',
    'total_usage_count', 'total_usage_duration', 'total_errors'
]
master_table[fill_zero_cols] = master_table[fill_zero_cols].fillna(0)

# ==========================================
# 9. RATIO FEATURES
# ==========================================
master_table['error_rate']           = master_table['total_errors']        / (master_table['total_usage_count']  + 1)
master_table['ticket_per_month']     = master_table['total_tickets']       / (master_table['tenure_days'] / 30   + 1)
master_table['escalation_rate']      = master_table['total_escalations']   / (master_table['total_tickets']      + 1)
master_table['avg_session_duration'] = master_table['total_usage_duration']/ (master_table['total_usage_count']  + 1)
master_table['error_per_mrr']        = master_table['total_errors']        / (master_table['total_mrr']          + 1)

# ==========================================
# 10. ENCODE CATEGORICAL COLUMNS
# ==========================================
# account_name ve signup_date artık işimize yaramıyor
master_table = master_table.drop(columns=['account_name', 'signup_date'])

model_type = 'tree'

# Nominal kategorikler → One-Hot
if model_type == 'linear':
    master_table = pd.get_dummies(master_table, columns=['industry', 'country', 'referral_source'], drop_first=True)
else:       
    master_table = pd.get_dummies(master_table, columns=['industry', 'country', 'referral_source'], drop_first=False)
                 

# Ordinal kategorik → Manuel sırala (plan_tier'ın doğal bir sırası var)
plan_tier_map = {'Basic': 0, 'Pro': 1, 'Enterprise': 2}
master_table['plan_tier'] = master_table['plan_tier'].map(plan_tier_map)

# Boolean kolonları int'e çevir
bool_cols = ['is_trial', 'churn_flag', 'provided_satisfaction_score']
master_table[bool_cols] = master_table[bool_cols].astype(int)

# ==========================================
# 11. FINAL CHECKS
# ==========================================
print("Shape:", master_table.shape)
print("\nChurn Rate:")
print(master_table['churn_flag'].value_counts(normalize=True).round(3))
print("\nKalan NaN:")
print(master_table.isnull().sum()[master_table.isnull().sum() > 0])
print("\nObject dtype kalan sütun:")
print(master_table.dtypes[master_table.dtypes == 'object'])

# ==========================================
# 12. SAVE
# ==========================================
master_table.to_csv(f"{processed_path}/master_features_final.csv", index=False)
print("\n--- MASTER TABLE SAVED ---")
print(master_table.head(3))

Shape: (500, 37)

Churn Rate:
churn_flag
0    0.78
1    0.22
Name: proportion, dtype: float64

Kalan NaN:
Series([], dtype: int64)

Object dtype kalan sütun:
account_id    object
dtype: object

--- MASTER TABLE SAVED ---
  account_id  plan_tier  seats  is_trial  churn_flag  total_tickets  \
0   A-2e4581          0      9         0           0            2.0   
1   A-43a9e3          0     18         0           1            3.0   
2   A-0a282f          0      1         0           0            3.0   

   avg_satisfaction  total_escalations  total_mrr  total_upgrades  ...  \
0          3.000000                0.0      12603               1  ...   
1          4.000000                0.0      10004               3  ...   
2          4.666667                0.0      18286               1  ...   

   country_DE  country_FR  country_IN  country_UK  country_US  \
0       False       False       False       False        True   
1       False       False        True       False       False   
2 